In [ ]:
import os
print(os.listdir("data/unsplash"))

In [ ]:
import faiss
import pandas as pd
import numpy as np

In [ ]:
index = faiss.read_index("data/unsplash/index.faiss")
meta = pd.read_csv("data/unsplash/metadata.csv")

print(index.ntotal, len(meta))

In [ ]:
from transformers import CLIPModel, CLIPProcessor
import torch

MODEL_NAME = "openai/clip-vit-large-patch14"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model.eval()

In [ ]:
def search(query_vec, k=5):
    query_vec = query_vec.reshape(1, -1).astype("float32")

    scores, indices = index.search(query_vec, k)

    results = []
    for i, idx in enumerate(indices[0]):
        row = meta.iloc[idx]

        results.append({
            "id": row["id"],
            "url": row["url"],
            "score": float(scores[0][i])
        })

    return results

In [ ]:
from models.embedder import embed_text, embed_image, embed_query, embed_multimodal

In [ ]:
results = search(embed_text("snow mountains"), k=5)
results

In [ ]:
from PIL import Image
import requests
from io import BytesIO

def load_image_from_url(url):
    response = requests.get(url)
    return Image.open(BytesIO(response.content)).convert("RGB")

test_img = load_image_from_url(meta.iloc[0]["url"])

results = search(embed_image(test_img), k=5)
results

In [ ]:
def show_results(query_img, results):
    import matplotlib.pyplot as plt

    plt.figure(figsize=(15, 5))

    # show query image first
    plt.subplot(1, len(results)+1, 1)
    plt.imshow(query_img)
    plt.title("Query")
    plt.axis("off")

    # show results
    for i, r in enumerate(results):
        img = load_image_from_url(r["url"])

        plt.subplot(1, len(results)+1, i+2)
        plt.imshow(img)
        plt.title(f"{r['score']:.2f}")
        plt.axis("off")

    plt.show()

In [ ]:
query_img = load_image_from_url(meta.iloc[0]["url"])
results = search(embed_image(query_img), k=5)

show_results(query_img, results)

In [ ]:
query_vec = embed_multimodal(
    image=query_img,
    text="sunset",
    alpha=0.7
)

results = search(query_vec)
show_results(query_img, results)